# Notebook 02 — Data Quality & Linkage

## Mục tiêu

Kiểm tra chất lượng metadata và xây dựng cơ sở để liên kết các bảng mô tả tổn thương với dữ liệu DICOM của CBIS-DDSM.

## Phạm vi

1. Xác nhận các tệp đầu vào.
2. Khảo sát schema và khóa định danh của các CSV.
3. Kiểm tra trùng lặp và giá trị thiếu.
4. Xác định quy tắc liên kết CSV, metadata và DICOM dựa trên dữ liệu thực tế.
5. Ghi nhận các trường hợp liên kết thành công, chưa thành công và nguyên nhân.

**Nguyên tắc:** Không sửa hoặc xóa dữ liệu gốc; không giả định quan hệ giữa ảnh toàn cảnh, ảnh crop, ROI mask và nhãn tổn thương trước khi kiểm tra.

In [44]:
from pathlib import Path

# Cấu hình đường dẫn
PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")

DATA_DIR = PROJECT_ROOT / "Data"
METADATA_DIR = DATA_DIR / "metadata"
RAWDATA_DIR = DATA_DIR / "rawdata"
RAW_METADATA_DIR = RAWDATA_DIR / "metadata"
IMAGE_ROOT = RAWDATA_DIR / "cbis_ddsm"

# Các tệp đầu vào
CSV_FILES = {
    "calc_test": METADATA_DIR / "calc_case_description_test_set.csv",
    "calc_train": METADATA_DIR / "calc_case_description_train_set.csv",
    "mass_test": METADATA_DIR / "mass_case_description_test_set.csv",
    "mass_train": METADATA_DIR / "mass_case_description_train_set.csv",
    "metadata": RAW_METADATA_DIR / "metadata.csv",
}

print("kiểm tra đầu vào")
print("=" * 60)

for name, path in CSV_FILES.items():
    status = "OK" if path.is_file() else "MISSING"
    print(f"[{status}] {name}: {path}")

print(
    f"[{'OK' if IMAGE_ROOT.is_dir() else 'MISSING'}] "
    f"IMAGE_ROOT: {IMAGE_ROOT}"
)

kiểm tra đầu vào
[OK] calc_test: C:\Breast-Cancer-CBIS-DDSM\Data\metadata\calc_case_description_test_set.csv
[OK] calc_train: C:\Breast-Cancer-CBIS-DDSM\Data\metadata\calc_case_description_train_set.csv
[OK] mass_test: C:\Breast-Cancer-CBIS-DDSM\Data\metadata\mass_case_description_test_set.csv
[OK] mass_train: C:\Breast-Cancer-CBIS-DDSM\Data\metadata\mass_case_description_train_set.csv
[OK] metadata: C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\metadata\metadata.csv
[OK] IMAGE_ROOT: C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm


In [45]:
for name, path in CSV_FILES.items():
    df = pd.read_csv(path, low_memory=False)

    print(f"\n{'=' * 50}")
    print(f"{name}: {df.shape[0]:,} dòng × {df.shape[1]} cột")
    print("Cột:", df.columns.tolist())
    print("Tổng giá trị thiếu:", int(df.isna().sum().sum()))
    print("Cột thiếu:")
    print(df.isna().sum()[df.isna().sum() > 0].to_dict())


calc_test: 326 dòng × 14 cột
Cột: ['patient_id', 'breast density', 'left or right breast', 'image view', 'abnormality id', 'abnormality type', 'calc type', 'calc distribution', 'assessment', 'pathology', 'subtlety', 'image file path', 'cropped image file path', 'ROI mask file path']
Tổng giá trị thiếu: 67
Cột thiếu:
{'calc type': 4, 'calc distribution': 63}

calc_train: 1,546 dòng × 14 cột
Cột: ['patient_id', 'breast density', 'left or right breast', 'image view', 'abnormality id', 'abnormality type', 'calc type', 'calc distribution', 'assessment', 'pathology', 'subtlety', 'image file path', 'cropped image file path', 'ROI mask file path']
Tổng giá trị thiếu: 396
Cột thiếu:
{'calc type': 20, 'calc distribution': 376}

mass_test: 378 dòng × 14 cột
Cột: ['patient_id', 'breast_density', 'left or right breast', 'image view', 'abnormality id', 'abnormality type', 'mass shape', 'mass margins', 'assessment', 'pathology', 'subtlety', 'image file path', 'cropped image file path', 'ROI mask fil

In [46]:
# Kiểm tra trùng lặp và khóa định danh

for name, path in CSV_FILES.items():
    df = pd.read_csv(path, low_memory=False)

    print(f"\n--- {name} ---")
    print(f"Số dòng: {len(df):,}")
    print(f"Số dòng trùng hoàn toàn: {df.duplicated().sum():,}")

    if name != "metadata":
        key_cols = [
            "patient_id",
            "left or right breast",
            "image view",
            "abnormality id",
        ]

        print("\nKhóa ứng viên:")
        for col in key_cols:
            if col in df.columns:
                print(
                    f"{col}: {df[col].nunique(dropna=False):,} "
                    f"giá trị phân biệt"
                )

        path_cols = [
            "image file path",
            "cropped image file path",
            "ROI mask file path",
        ]

        print("\nĐường dẫn ảnh:")
        for col in path_cols:
            if col in df.columns:
                print(
                    f"{col}: "
                    f"{df[col].notna().sum():,}/{len(df):,} có giá trị"
                )

    else:
        for col in ["PatientID", "StudyInstanceUID", "SeriesInstanceUID"]:
            if col in df.columns:
                print(
                    f"{col}: {df[col].nunique(dropna=False):,} "
                    f"giá trị phân biệt"
                )


--- calc_test ---
Số dòng: 326
Số dòng trùng hoàn toàn: 0

Khóa ứng viên:
patient_id: 151 giá trị phân biệt
left or right breast: 2 giá trị phân biệt
image view: 2 giá trị phân biệt
abnormality id: 5 giá trị phân biệt

Đường dẫn ảnh:
image file path: 326/326 có giá trị
cropped image file path: 326/326 có giá trị
ROI mask file path: 326/326 có giá trị

--- calc_train ---
Số dòng: 1,546
Số dòng trùng hoàn toàn: 0

Khóa ứng viên:
patient_id: 602 giá trị phân biệt
left or right breast: 2 giá trị phân biệt
image view: 2 giá trị phân biệt
abnormality id: 7 giá trị phân biệt

Đường dẫn ảnh:
image file path: 1,546/1,546 có giá trị
cropped image file path: 1,546/1,546 có giá trị
ROI mask file path: 1,546/1,546 có giá trị

--- mass_test ---
Số dòng: 378
Số dòng trùng hoàn toàn: 0

Khóa ứng viên:
patient_id: 201 giá trị phân biệt
left or right breast: 2 giá trị phân biệt
image view: 2 giá trị phân biệt
abnormality id: 4 giá trị phân biệt

Đường dẫn ảnh:
image file path: 378/378 có giá trị
croppe

In [47]:
import pandas as pd

print("KIỂM TRA KHÓA ỨNG VIÊN")
print("=" * 70)

cac_khoa = [
    "patient_id",
    "left or right breast",
    "image view",
    "abnormality id",
]

for ten in ["calc_test", "calc_train", "mass_test", "mass_train"]:
    df = pd.read_csv(CSV_FILES[ten], low_memory=False)

    khoa_hop_le = [cot for cot in cac_khoa if cot in df.columns]

    print(f"\n--- {ten} ---")

    # Kiểm tra từng cột riêng lẻ
    for cot in khoa_hop_le:
        so_dong_lap = df[cot].duplicated().sum()
        print(f"{cot}: {so_dong_lap:,} dòng có giá trị lặp")

    # Kiểm tra tổ hợp các cột
    so_dong_lap = df.duplicated(subset=khoa_hop_le).sum()
    print(f"Số dòng lặp theo tổ hợp khóa: {so_dong_lap:,}")

    # Hiển thị một số trường hợp lặp để khảo sát
    dong_lap = df[df.duplicated(subset=khoa_hop_le, keep=False)]

    if not dong_lap.empty:
        print("Một số trường hợp cần xem xét:")
        print(dong_lap[khoa_hop_le].head(10).to_string(index=False))
    else:
        print("Không có tổ hợp khóa nào bị lặp.")


KIỂM TRA KHÓA ỨNG VIÊN

--- calc_test ---
patient_id: 175 dòng có giá trị lặp
left or right breast: 324 dòng có giá trị lặp
image view: 324 dòng có giá trị lặp
abnormality id: 321 dòng có giá trị lặp
Số dòng lặp theo tổ hợp khóa: 0
Không có tổ hợp khóa nào bị lặp.

--- calc_train ---
patient_id: 944 dòng có giá trị lặp
left or right breast: 1,544 dòng có giá trị lặp
image view: 1,544 dòng có giá trị lặp
abnormality id: 1,539 dòng có giá trị lặp
Số dòng lặp theo tổ hợp khóa: 0
Không có tổ hợp khóa nào bị lặp.

--- mass_test ---
patient_id: 177 dòng có giá trị lặp
left or right breast: 376 dòng có giá trị lặp
image view: 376 dòng có giá trị lặp
abnormality id: 374 dòng có giá trị lặp
Số dòng lặp theo tổ hợp khóa: 0
Không có tổ hợp khóa nào bị lặp.

--- mass_train ---
patient_id: 627 dòng có giá trị lặp
left or right breast: 1,316 dòng có giá trị lặp
image view: 1,316 dòng có giá trị lặp
abnormality id: 1,312 dòng có giá trị lặp
Số dòng lặp theo tổ hợp khóa: 0
Không có tổ hợp khóa nào bị 

In [48]:
from pathlib import Path
import pandas as pd

print("KIỂM TRA ĐƯỜNG DẪN ẢNH TRONG CÁC BẢNG MÔ TẢ")
print("=" * 75)

cac_cot_duong_dan = [
    "image file path",
    "cropped image file path",
    "ROI mask file path",
]

for ten in ["calc_test", "calc_train", "mass_test", "mass_train"]:
    df = pd.read_csv(CSV_FILES[ten], low_memory=False)

    print(f"\n--- {ten} ({len(df):,} dòng) ---")

    for cot in cac_cot_duong_dan:
        if cot not in df.columns:
            continue

        du_lieu = df[cot].dropna().astype(str).str.strip()
        du_lieu = du_lieu[du_lieu.ne("")]

        print(f"\n{cot}:")
        print(f"  Có đường dẫn: {len(du_lieu):,}/{len(df):,}")

        # Kiểm tra trực tiếp đường dẫn nếu tồn tại trên máy
        ton_tai = du_lieu.map(lambda x: Path(x).is_file())
        print(f"  Đường dẫn hiện có trỏ trực tiếp đến tệp: "
              f"{ton_tai.sum():,}/{len(du_lieu):,}")

        # Xem một vài đường dẫn gốc để khảo sát cấu trúc
        print("  Ví dụ:")
        for gia_tri in du_lieu.head(3):
            print(f"    {gia_tri}")


KIỂM TRA ĐƯỜNG DẪN ẢNH TRONG CÁC BẢNG MÔ TẢ

--- calc_test (326 dòng) ---

image file path:
  Có đường dẫn: 326/326
  Đường dẫn hiện có trỏ trực tiếp đến tệp: 0/326
  Ví dụ:
    Calc-Test_P_00038_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.85935434310203356712688695661986996009/1.3.6.1.4.1.9590.100.1.2.374115997511889073021386151921807063992/000000.dcm
    Calc-Test_P_00038_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.384159464510350889125645400702639717613/1.3.6.1.4.1.9590.100.1.2.174390361112646747718661211471328897934/000000.dcm
    Calc-Test_P_00038_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.177706148911820252341905176394069228468/1.3.6.1.4.1.9590.100.1.2.263861248711313923336051913560309963304/000000.dcm

cropped image file path:
  Có đường dẫn: 326/326
  Đường dẫn hiện có trỏ trực tiếp đến tệp: 0/326
  Ví dụ:
    Calc-Test_P_00038_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.161465562211359959230647609981488894942/1.3.6.1.4.1.9590.100.1.2.419081637812053404913157930753972718515/000001.dcm
    Calc-Test_P_00038_LEFT_MLO_

### Tổ hợp đang kiểm tra gồm patient_id, left or right breast, image view và abnormality id.
- Không trùng tổ hợp: không có hai dòng nào trong từng bảng có cùng giá trị ở cả 4 cột này.
- Các cột riêng lẻ bị lặp: là điều bình thường, vì một bệnh nhân có thể có nhiều tổn thương hoặc nhiều ảnh.
- Chưa xác nhận khóa liên kết giữa CSV và DICOM: cần kiểm tra thêm đường dẫn ảnh và cách đối chiếu với metadata.csv.

In [49]:
# Khảo sát cấu trúc metadata và định danh DICOM

print("KHẢO SÁT THÔNG TIN LIÊN KẾT")
print("=" * 75)

metadata = pd.read_csv(CSV_FILES["metadata"], low_memory=False)

cac_cot_can_khao_sat = [
    "PatientID",
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "SeriesDescription",
    "DownloadURL",
    "S5cmdManifestPath",
    "OriginalS5cmdURI",
]

for cot in cac_cot_can_khao_sat:
    if cot not in metadata.columns:
        print(f"\n{cot}: Không có trong metadata")
        continue

    print(f"\n{cot}")
    print(f"  Số giá trị khác nhau: {metadata[cot].nunique(dropna=True):,}")
    print(f"  Số giá trị thiếu: {metadata[cot].isna().sum():,}")
    print("  Ba giá trị mẫu:")
    print(metadata[cot].dropna().head(3).to_string(index=False))

print("\nCác cột đường dẫn trong CSV mô tả tổn thương:")
for ten in ["calc_test", "calc_train", "mass_test", "mass_train"]:
    df = pd.read_csv(CSV_FILES[ten], low_memory=False)
    print(f"\n{ten}")
    for cot in [
        "patient_id",
        "image file path",
        "cropped image file path",
        "ROI mask file path",
    ]:
        if cot in df.columns:
            print(f"\n{cot}:")
            print(df[cot].dropna().head(2).to_string(index=False))


KHẢO SÁT THÔNG TIN LIÊN KẾT

PatientID
  Số giá trị khác nhau: 6,671
  Số giá trị thiếu: 0
  Ba giá trị mẫu:
Calc-Test_P_01004_LEFT_MLO_1
Calc-Test_P_00038_LEFT_MLO_1
      P_00041_LEFT_MLO_2.dcm

StudyInstanceUID
  Số giá trị khác nhau: 6,775
  Số giá trị thiếu: 0
  Ba giá trị mẫu:
1.3.6.1.4.1.9590.100.1.2.2651982305124555095196...
1.3.6.1.4.1.9590.100.1.2.2911219961314313853538...
1.3.6.1.4.1.9590.100.1.2.3729622900110685890083...

SeriesInstanceUID
  Số giá trị khác nhau: 6,775
  Số giá trị thiếu: 0
  Ba giá trị mẫu:
1.3.6.1.4.1.9590.100.1.2.2944450479124070300122...
1.3.6.1.4.1.9590.100.1.2.1886139557101704178030...
1.3.6.1.4.1.9590.100.1.2.3994662582126469320184...

SeriesDescription
  Số giá trị khác nhau: 3
  Số giá trị thiếu: 246
  Ba giá trị mẫu:
cropped images
cropped images
cropped images

DownloadURL
  Số giá trị khác nhau: 6,775
  Số giá trị thiếu: 0
  Ba giá trị mẫu:
s3://idc-open-data/b4dd3465-76ef-4cb3-baa8-f919...
s3://idc-open-data/620b14e7-eeb0-4570-869a-3109...
s3:/

In [50]:
# Lập chỉ mục DICOM từ dữ liệu thực tế

import pydicom

print("LẬP CHỈ MỤC TỆP DICOM")
print("=" * 75)

duong_dan_dicom = [
    p for p in IMAGE_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() == ".dcm"
]

print(f"Tổng số tệp DICOM: {len(duong_dan_dicom):,}")

# Chỉ đọc header, không giải mã pixel
dong_chi_muc = []

for i, tep in enumerate(duong_dan_dicom, start=1):
    try:
        ds = pydicom.dcmread(
            str(tep),
            stop_before_pixels=True,
            force=True,
        )

        dong_chi_muc.append({
            "file_path": str(tep),
            "relative_path": str(tep.relative_to(IMAGE_ROOT)),
            "PatientID": str(getattr(ds, "PatientID", "")),
            "StudyInstanceUID": str(getattr(ds, "StudyInstanceUID", "")),
            "SeriesInstanceUID": str(getattr(ds, "SeriesInstanceUID", "")),
            "SOPInstanceUID": str(getattr(ds, "SOPInstanceUID", "")),
            "SeriesDescription": str(getattr(ds, "SeriesDescription", "")),
        })

    except Exception as e:
        dong_chi_muc.append({
            "file_path": str(tep),
            "relative_path": str(tep.relative_to(IMAGE_ROOT)),
            "PatientID": "",
            "StudyInstanceUID": "",
            "SeriesInstanceUID": "",
            "SOPInstanceUID": "",
            "SeriesDescription": "",
        })

    if i % 1000 == 0:
        print(f"Đã kiểm tra header: {i:,}/{len(duong_dan_dicom):,}")

dicom_index = pd.DataFrame(dong_chi_muc)

print("\nTóm tắt chỉ mục:")
print(f"Số dòng chỉ mục: {len(dicom_index):,}")
print(f"Thiếu SeriesInstanceUID: "
      f"{dicom_index['SeriesInstanceUID'].eq('').sum():,}")
print(f"Thiếu SOPInstanceUID: "
      f"{dicom_index['SOPInstanceUID'].eq('').sum():,}")

print("\nSố SeriesInstanceUID phân biệt:",
      dicom_index.loc[
          dicom_index["SeriesInstanceUID"].ne(""),
          "SeriesInstanceUID"
      ].nunique())

print("\nSố SOPInstanceUID phân biệt:",
      dicom_index.loc[
          dicom_index["SOPInstanceUID"].ne(""),
          "SOPInstanceUID"
      ].nunique())


LẬP CHỈ MỤC TỆP DICOM
Tổng số tệp DICOM: 10,239
Đã kiểm tra header: 1,000/10,239
Đã kiểm tra header: 2,000/10,239
Đã kiểm tra header: 3,000/10,239
Đã kiểm tra header: 4,000/10,239
Đã kiểm tra header: 5,000/10,239
Đã kiểm tra header: 6,000/10,239
Đã kiểm tra header: 7,000/10,239
Đã kiểm tra header: 8,000/10,239
Đã kiểm tra header: 9,000/10,239
Đã kiểm tra header: 10,000/10,239

Tóm tắt chỉ mục:
Số dòng chỉ mục: 10,239
Thiếu SeriesInstanceUID: 0
Thiếu SOPInstanceUID: 0

Số SeriesInstanceUID phân biệt: 6775

Số SOPInstanceUID phân biệt: 10239


In [51]:
# ĐỐI CHIẾU ĐỊNH DANH SERIES")
# Mục tiêu: xác nhận quan hệ giữa metadata và chỉ mục DICOM

# Đọc metadata đã có trong bộ dữ liệu
metadata_df = pd.read_csv(CSV_FILES["metadata"], dtype=str)

# Chuẩn hóa kiểu dữ liệu và khoảng trắng của khóa định danh
metadata_df["SeriesInstanceUID"] = (
    metadata_df["SeriesInstanceUID"].astype("string").str.strip()
)

dicom_index["SeriesInstanceUID"] = (
    dicom_index["SeriesInstanceUID"].astype("string").str.strip()
)

# Lập tập khóa Series hợp lệ
series_metadata = set(
    metadata_df["SeriesInstanceUID"].dropna()
)
series_dicom = set(
    dicom_index["SeriesInstanceUID"].dropna()
)

# Đối chiếu hai nguồn
series_trung_khop = series_metadata & series_dicom
series_chi_co_metadata = series_metadata - series_dicom
series_chi_co_dicom = series_dicom - series_metadata

print("=" * 75)
print("ĐỐI CHIẾU SERIES GIỮA METADATA VÀ DICOM")
print("=" * 75)
print(f"Số Series trong metadata: {len(series_metadata):,}")
print(f"Số Series trong chỉ mục DICOM: {len(series_dicom):,}")
print(f"Số Series trùng khớp: {len(series_trung_khop):,}")
print(f"Số Series chỉ có trong metadata: {len(series_chi_co_metadata):,}")
print(f"Số Series chỉ có trong DICOM: {len(series_chi_co_dicom):,}")

# Kiểm tra khóa Series có lặp trong mỗi nguồn hay không
print("\nKIỂM TRA SỐ DÒNG TRÊN MỖI SERIES")
print(
    "Metadata có SeriesInstanceUID lặp:",
    int(metadata_df["SeriesInstanceUID"].duplicated().sum())
)

so_tap_tin_moi_series = (
    dicom_index.groupby("SeriesInstanceUID", dropna=True)
    .size()
)

print(
    "Số Series có đúng 1 tệp DICOM:",
    int((so_tap_tin_moi_series == 1).sum())
)
print(
    "Số Series có nhiều hơn 1 tệp DICOM:",
    int((so_tap_tin_moi_series > 1).sum())
)

# Lưu các kết quả đối chiếu trong bộ nhớ để dùng ở bước sau
ket_qua_doi_chieu_series = {
    "series_metadata": series_metadata,
    "series_dicom": series_dicom,
    "series_trung_khop": series_trung_khop,
    "series_chi_co_metadata": series_chi_co_metadata,
    "series_chi_co_dicom": series_chi_co_dicom,
    "so_tap_tin_moi_series": so_tap_tin_moi_series,
}


ĐỐI CHIẾU SERIES GIỮA METADATA VÀ DICOM
Số Series trong metadata: 6,775
Số Series trong chỉ mục DICOM: 6,775
Số Series trùng khớp: 6,775
Số Series chỉ có trong metadata: 0
Số Series chỉ có trong DICOM: 0

KIỂM TRA SỐ DÒNG TRÊN MỖI SERIES
Metadata có SeriesInstanceUID lặp: 0
Số Series có đúng 1 tệp DICOM: 3311
Số Series có nhiều hơn 1 tệp DICOM: 3464


In [52]:
# PHÂN TÍCH CẤU TRÚC ĐƯỜNG DẪN
# Mục tiêu: tìm quy tắc liên kết từ CSV đến tệp DICOM

from pathlib import PurePosixPath, PureWindowsPath
from collections import Counter
import re

CAC_COT_DUONG_DAN = [
    "image file path",
    "cropped image file path",
    "ROI mask file path",
]

CAC_BANG = [
    "calc_test",
    "calc_train",
    "mass_test",
    "mass_train",
]

print("=" * 75)
print("PHÂN TÍCH CẤU TRÚC ĐƯỜNG DẪN TRONG BỐN CSV")
print("=" * 75)

for ten_bang in CAC_BANG:
    df = pd.read_csv(CSV_FILES[ten_bang], dtype=str)

    print(f"\n{'=' * 75}")
    print(f"BẢNG: {ten_bang} | Số dòng: {len(df):,}")
    print("=" * 75)

    for ten_cot in CAC_COT_DUONG_DAN:
        if ten_cot not in df.columns:
            print(f"\n{ten_cot}: KHÔNG CÓ CỘT")
            continue

        gia_tri = df[ten_cot].dropna().astype(str).str.strip()
        gia_tri = gia_tri[gia_tri.ne("")]

        print(f"\nCột: {ten_cot}")
        print(f"Số đường dẫn không rỗng: {len(gia_tri):,}")
        print(f"Số đường dẫn phân biệt: {gia_tri.nunique():,}")

        # Thống kê số thành phần của đường dẫn
        so_thanh_phan = gia_tri.map(
            lambda p: len([
                x for x in re.split(r"[/\\]+", p)
                if x.strip()
            ])
        )
        print("Số thành phần đường dẫn:")
        print(so_thanh_phan.value_counts().sort_index().to_dict())

        # Xác định dạng thành phần cuối
        ten_cuoi = gia_tri.map(
            lambda p: re.split(r"[/\\]+", p.rstrip("/\\"))[-1]
        )

        print(
            "Thành phần cuối có đuôi .dcm:",
            int(ten_cuoi.str.lower().str.endswith(".dcm").sum())
        )

        # Kiểm tra thành phần có hình dạng UID
        cac_thanh_phan = gia_tri.map(
            lambda p: [
                x for x in re.split(r"[/\\]+", p)
                if x.strip()
            ]
        )

        so_duong_dan_co_uid = int(
            cac_thanh_phan.map(
                lambda parts: any(
                    re.fullmatch(r"\d+(?:\.\d+)+", x) is not None
                    for x in parts
                )
            ).sum()
        )
        print(
            "Đường dẫn có thành phần dạng UID:",
            so_duong_dan_co_uid
        )

        # In tối đa 3 mẫu thực tế để nhận diện cấu trúc
        print("Mẫu đường dẫn:")
        for mau in gia_tri.drop_duplicates().head(3):
            print(f"  {mau}")

print("\n" + "=" * 75)
print("ĐÃ HOÀN TẤT PHÂN TÍCH CẤU TRÚC")

PHÂN TÍCH CẤU TRÚC ĐƯỜNG DẪN TRONG BỐN CSV

BẢNG: calc_test | Số dòng: 326

Cột: image file path
Số đường dẫn không rỗng: 326
Số đường dẫn phân biệt: 284
Số thành phần đường dẫn:
{4: 326}
Thành phần cuối có đuôi .dcm: 326
Đường dẫn có thành phần dạng UID: 326
Mẫu đường dẫn:
  Calc-Test_P_00038_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.85935434310203356712688695661986996009/1.3.6.1.4.1.9590.100.1.2.374115997511889073021386151921807063992/000000.dcm
  Calc-Test_P_00038_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.384159464510350889125645400702639717613/1.3.6.1.4.1.9590.100.1.2.174390361112646747718661211471328897934/000000.dcm
  Calc-Test_P_00038_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.177706148911820252341905176394069228468/1.3.6.1.4.1.9590.100.1.2.263861248711313923336051913560309963304/000000.dcm

Cột: cropped image file path
Số đường dẫn không rỗng: 326
Số đường dẫn phân biệt: 326
Số thành phần đường dẫn:
{4: 326}
Thành phần cuối có đuôi .dcm: 326
Đường dẫn có thành phần dạng UID: 326
Mẫu đường dẫn:
  Calc-Te

- Kết quả này xác nhận được cấu trúc đường dẫn của CSV: mỗi đường dẫn có 4 thành phần, gồm tên thư mục đầu tiên, hai thành phần dạng UID và tên tệp DICOM ở cuối.

In [53]:
# XÁC MINH UID VÀ ĐỐI CHIẾU VỚI DICOM
# Sử dụng chỉ mục đã tạo, không đọc lại header DICOM

CAC_BANG = ["calc_test", "calc_train", "mass_test", "mass_train"]

CAC_COT_DUONG_DAN = [
    "image file path",
    "cropped image file path",
    "ROI mask file path",
]

# Kiểm tra các cột cần thiết trong chỉ mục hiện có
CAC_COT_CAN_CO = {
    "file_path",
    "StudyInstanceUID",
    "SeriesInstanceUID",
}

if not CAC_COT_CAN_CO.issubset(dicom_index.columns):
    raise ValueError(
        "Chỉ mục DICOM thiếu cột cần thiết: "
        f"{CAC_COT_CAN_CO - set(dicom_index.columns)}"
    )

# Tạo tập UID từ chỉ mục DICOM
tap_study_uid = set(
    dicom_index["StudyInstanceUID"].dropna().astype(str).str.strip()
)
tap_series_uid = set(
    dicom_index["SeriesInstanceUID"].dropna().astype(str).str.strip()
)

# Lập chỉ mục các cặp Study + Series đã xác nhận trong DICOM
dicom_theo_cap_uid = {}

for _, dong in dicom_index.iterrows():
    study_uid = str(dong["StudyInstanceUID"]).strip()
    series_uid = str(dong["SeriesInstanceUID"]).strip()

    if study_uid and series_uid:
        dicom_theo_cap_uid.setdefault(
            (study_uid, series_uid), []
        ).append(str(dong["file_path"]))

def tach_duong_dan(gia_tri):
    """Tách các thành phần của đường dẫn theo dấu / hoặc \\."""
    if pd.isna(gia_tri):
        return []
    return [
        p.strip()
        for p in re.split(r"[/\\]+", str(gia_tri).strip())
        if p.strip()
    ]

def la_uid(gia_tri):
    return re.fullmatch(r"\d+(?:\.\d+)+", gia_tri) is not None

ket_qua_lien_ket = []

for ten_bang in CAC_BANG:
    df = pd.read_csv(CSV_FILES[ten_bang], dtype=str)

    for ten_cot in CAC_COT_DUONG_DAN:
        if ten_cot not in df.columns:
            continue

        for chi_so, gia_tri in df[ten_cot].items():
            thanh_phan = tach_duong_dan(gia_tri)

            if not thanh_phan:
                ket_qua_lien_ket.append({
                    "bang": ten_bang,
                    "cot": ten_cot,
                    "dong_csv": chi_so,
                    "duong_dan": gia_tri,
                    "so_uid": 0,
                    "vi_tri_1_la_study": False,
                    "vi_tri_2_la_series": False,
                    "cap_uid_ton_tai": False,
                    "ten_tep_khop": False,
                    "ket_qua": "Đường dẫn rỗng",
                })
                continue

            ten_tep = thanh_phan[-1]
            cac_uid = [p for p in thanh_phan if la_uid(p)]

            # Theo cấu trúc quan sát được: thành phần 2 và 3được kiểm tra như Study UID và Series UID.
            study_uid = thanh_phan[1] if len(thanh_phan) >= 4 else ""
            series_uid = thanh_phan[2] if len(thanh_phan) >= 4 else ""

            dung_study = study_uid in tap_study_uid
            dung_series = series_uid in tap_series_uid

            # Cặp UID chỉ được xác nhận khi cùng xuất hiện trong một hoặc nhiều tệp DICOM của chỉ mục.
            cac_tep_cung_cap = dicom_theo_cap_uid.get(
                (study_uid, series_uid), []
            )
            cap_uid_ton_tai = len(cac_tep_cung_cap) > 0

            # Chỉ so sánh tên tệp sau khi cặp UID đã khớp.
            ten_tep_khop = (
                cap_uid_ton_tai
                and any(
                    PureWindowsPath(p).name.lower() == ten_tep.lower()
                    or PurePosixPath(p.replace("\\", "/")).name.lower()
                    == ten_tep.lower()
                    for p in cac_tep_cung_cap
                )
            )

            if cap_uid_ton_tai and ten_tep_khop:
                trang_thai = "Khớp cặp UID và tên tệp"
            elif cap_uid_ton_tai:
                trang_thai = "Khớp cặp UID, chưa khớp tên tệp"
            elif dung_study and dung_series:
                trang_thai = "Hai UID tồn tại riêng lẻ, chưa khớp cùng cặp"
            else:
                trang_thai = "Chưa xác nhận được cặp UID"

            ket_qua_lien_ket.append({
                "bang": ten_bang,
                "cot": ten_cot,
                "dong_csv": chi_so,
                "duong_dan": gia_tri,
                "so_uid": len(cac_uid),
                "vi_tri_1_la_study": (
                    len(thanh_phan) >= 4
                    and thanh_phan[1] in tap_study_uid
                ),
                "vi_tri_2_la_series": (
                    len(thanh_phan) >= 4
                    and thanh_phan[2] in tap_series_uid
                ),
                "cap_uid_ton_tai": cap_uid_ton_tai,
                "ten_tep_khop": ten_tep_khop,
                "ket_qua": trang_thai,
            })

bang_ket_qua_lien_ket = pd.DataFrame(ket_qua_lien_ket)

print("=" * 75)
print("TỔNG HỢP ĐỐI CHIẾU UID VÀ TÊN TỆP")
print("=" * 75)

for ten_bang in CAC_BANG:
    for ten_cot in CAC_COT_DUONG_DAN:
        nhom = bang_ket_qua_lien_ket[
            (bang_ket_qua_lien_ket["bang"] == ten_bang)
            & (bang_ket_qua_lien_ket["cot"] == ten_cot)
        ]

        if nhom.empty:
            continue

        print(f"\n{ten_bang} | {ten_cot}")
        print(f"Số dòng kiểm tra: {len(nhom):,}")
        print(
            "Thành phần UID thứ nhất khớp Study UID:",
            int(nhom["vi_tri_1_la_study"].sum())
        )
        print(
            "Thành phần UID thứ hai khớp Series UID:",
            int(nhom["vi_tri_2_la_series"].sum())
        )
        print(
            "Khớp cặp Study + Series:",
            int(nhom["cap_uid_ton_tai"].sum())
        )
        print(
            "Khớp cả cặp UID và tên tệp:",
            int(nhom["ten_tep_khop"].sum())
        )
        print("Phân loại kết quả:")
        print(nhom["ket_qua"].value_counts().to_string())

print("\n" + "=" * 75)
print("HOÀN TẤT ĐỐI CHIẾU")
print("Dữ liệu gốc không bị thay đổi.")


TỔNG HỢP ĐỐI CHIẾU UID VÀ TÊN TỆP

calc_test | image file path
Số dòng kiểm tra: 326
Thành phần UID thứ nhất khớp Study UID: 326
Thành phần UID thứ hai khớp Series UID: 326
Khớp cặp Study + Series: 326
Khớp cả cặp UID và tên tệp: 0
Phân loại kết quả:
ket_qua
Khớp cặp UID, chưa khớp tên tệp    326

calc_test | cropped image file path
Số dòng kiểm tra: 326
Thành phần UID thứ nhất khớp Study UID: 326
Thành phần UID thứ hai khớp Series UID: 326
Khớp cặp Study + Series: 326
Khớp cả cặp UID và tên tệp: 0
Phân loại kết quả:
ket_qua
Khớp cặp UID, chưa khớp tên tệp    326

calc_test | ROI mask file path
Số dòng kiểm tra: 326
Thành phần UID thứ nhất khớp Study UID: 326
Thành phần UID thứ hai khớp Series UID: 326
Khớp cặp Study + Series: 326
Khớp cả cặp UID và tên tệp: 0
Phân loại kết quả:
ket_qua
Khớp cặp UID, chưa khớp tên tệp    326

calc_train | image file path
Số dòng kiểm tra: 1,546
Thành phần UID thứ nhất khớp Study UID: 1546
Thành phần UID thứ hai khớp Series UID: 1546
Khớp cặp Study + Se

In [54]:
# KIỂM TRA TÊN TỆP TRONG SERIES ĐÃ KHỚP UID
# Mục tiêu: tìm nguyên nhân kết quả khớp tên tệp bằng 0

def lay_ten_tep(duong_dan):
    duong_dan = str(duong_dan).replace("\\", "/")
    return PurePosixPath(duong_dan).name.lower()

def tach_thanh_phan(duong_dan):
    return [
        p.strip()
        for p in re.split(r"[/\\]+", str(duong_dan).strip())
        if p.strip()
    ]

# Tạo ánh xạ cặp UID sang các tệp đã lập chỉ mục
tep_theo_cap_uid = {}

for _, dong in dicom_index.iterrows():
    study_uid = str(dong["StudyInstanceUID"]).strip()
    series_uid = str(dong["SeriesInstanceUID"]).strip()
    duong_dan_thuc_te = str(dong["file_path"])

    if study_uid and series_uid:
        khoa = (study_uid, series_uid)
        tep_theo_cap_uid.setdefault(khoa, []).append(
            duong_dan_thuc_te
        )

print("=" * 75)
print("KIỂM TRA TÊN TỆP THỰC TẾ")
print("=" * 75)

# In một số đường dẫn thật trong chỉ mục để kiểm tra cấu trúc
print("\nMỘT SỐ ĐƯỜNG DẪN TRONG CHỈ MỤC DICOM:")
for p in dicom_index["file_path"].head(5):
    print(" ", p)

# Đối chiếu tên tệp cuối đường dẫn với các tệp thuộc đúng cặp UID
tong_luot = 0
tong_khop_ten = 0
tong_co_cap_uid = 0
cac_truong_hop_lech = []

for ten_bang in ["calc_test", "calc_train", "mass_test", "mass_train"]:
    df = pd.read_csv(CSV_FILES[ten_bang], dtype=str)

    for ten_cot in [
        "image file path",
        "cropped image file path",
        "ROI mask file path",
    ]:
        for chi_so, duong_dan_csv in df[ten_cot].items():
            if pd.isna(duong_dan_csv) or not str(duong_dan_csv).strip():
                continue

            parts = tach_thanh_phan(duong_dan_csv)

            if len(parts) < 4:
                continue

            study_uid = parts[1]
            series_uid = parts[2]
            ten_tep_csv = lay_ten_tep(parts[-1])

            cac_tep_thuc_te = tep_theo_cap_uid.get(
                (study_uid, series_uid), []
            )

            tong_luot += 1

            if cac_tep_thuc_te:
                tong_co_cap_uid += 1
                cac_ten_thuc_te = {
                    lay_ten_tep(p) for p in cac_tep_thuc_te
                }

                if ten_tep_csv in cac_ten_thuc_te:
                    tong_khop_ten += 1
                elif len(cac_truong_hop_lech) < 10:
                    cac_truong_hop_lech.append({
                        "bang": ten_bang,
                        "cot": ten_cot,
                        "dong_csv": chi_so,
                        "ten_tep_csv": parts[-1],
                        "so_tep_trong_series": len(cac_tep_thuc_te),
                        "ten_tep_thuc_te": [
                            PurePosixPath(
                                str(p).replace("\\", "/")
                            ).name
                            for p in cac_tep_thuc_te[:5]
                        ],
                    })

print("\nTỔNG HỢP:")
print(f"Tổng lượt đường dẫn: {tong_luot:,}")
print(f"Lượt có cặp UID trong chỉ mục: {tong_co_cap_uid:,}")
print(f"Lượt khớp tên tệp trong đúng Series: {tong_khop_ten:,}")
print(
    "Lượt chưa khớp tên tệp:",
    f"{tong_co_cap_uid - tong_khop_ten:,}"
)

print("\nTỐI ĐA 10 TRƯỜNG HỢP CHƯA KHỚP:")
for truong_hop in cac_truong_hop_lech:
    print("-" * 60)
    print(f"Bảng: {truong_hop['bang']}")
    print(f"Cột: {truong_hop['cot']}")
    print(f"Dòng CSV (chỉ số): {truong_hop['dong_csv']}")
    print(f"Tên tệp trong CSV: {truong_hop['ten_tep_csv']}")
    print(f"Số tệp trong Series: {truong_hop['so_tep_trong_series']}")
    print(f"Tên tệp thực tế (tối đa 5): {truong_hop['ten_tep_thuc_te']}")

print("\nChỉ đọc chỉ mục và CSV; không thay đổi dữ liệu gốc.")


KIỂM TRA TÊN TỆP THỰC TẾ

MỘT SỐ ĐƯỜNG DẪN TRONG CHỈ MỤC DICOM:
  C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Test_P_00038_LEFT_MLO_1\2017-08-29-11739\1-cropped images-88680\69e26641-50ca-4837-9e0e-1b9523b8848b.dcm
  C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Test_P_00038_LEFT_MLO_1\2017-08-29-11739\1-cropped images-88680\c959748f-9c7c-419b-8c9c-cb63bfe65e44.dcm
  C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Test_P_00038_RIGHT_CC_1\2017-08-29-83105\1-cropped images-37851\574954cb-fdd5-46b1-875a-11a9eda9d911.dcm
  C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Test_P_00038_RIGHT_CC_1\2017-08-29-83105\1-cropped images-37851\e251af59-58b8-4dc7-98f6-7f08828e885e.dcm
  C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_ddsm\Calc-Test_P_00038_RIGHT_MLO_2\2017-08-29-68463\1-cropped images-04551\8b49b8bd-1885-4d2e-a657-4e49ac70c598.dcm

TỔNG HỢP:
Tổng lượt đường dẫn: 10,704
Lượt có cặp UID trong chỉ mục: 10,704
Lượt khớp tên tệp trong đúng Series: 0
Lượt chưa

In [55]:
# KIỂM TRA KHẢ NĂNG LIÊN KẾT Ở CẤP TỆP DICOM

print("CÁC CỘT TRONG CHỈ MỤC DICOM:")
print(dicom_index.columns.tolist())

print("\nCÁC CỘT ĐỊNH DANH TRONG METADATA:")
print([
    cot for cot in metadata_df.columns
    if any(
        tu_khoa in cot.lower()
        for tu_khoa in [
            "patient", "study", "series", "sop", "instance",
            "description", "download"
        ]
    )
])

print("\nMỘT DÒNG CSV MẪU:")
df_mau = pd.read_csv(CSV_FILES["calc_test"], dtype=str)
print(df_mau.loc[0, [
    "patient_id",
    "image file path",
    "cropped image file path",
    "ROI mask file path"
]].to_string())

print("\nMỘT DÒNG DICOM MẪU:")
print(
    dicom_index.loc[0, [
        "PatientID",
        "StudyInstanceUID",
        "SeriesInstanceUID",
        "SOPInstanceUID",
        "file_path"
    ]].to_string()
)


CÁC CỘT TRONG CHỈ MỤC DICOM:
['file_path', 'relative_path', 'PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'SOPInstanceUID', 'SeriesDescription']

CÁC CỘT ĐỊNH DANH TRONG METADATA:
['PatientID', 'StudyInstanceUID', 'StudyDate', 'SeriesInstanceUID', 'SeriesDescription', 'SeriesNumber', 'SOPClassUID', 'sop_class_name', 'crdc_series_uuid', 'DownloadURL', 'DownloadSource']

MỘT DÒNG CSV MẪU:
patient_id                                                           P_00038
image file path            Calc-Test_P_00038_LEFT_CC/1.3.6.1.4.1.9590.100...
cropped image file path    Calc-Test_P_00038_LEFT_CC_1/1.3.6.1.4.1.9590.1...
ROI mask file path         Calc-Test_P_00038_LEFT_CC_1/1.3.6.1.4.1.9590.1...

MỘT DÒNG DICOM MẪU:
PatientID                                 Calc-Test_P_00038_LEFT_MLO_1
StudyInstanceUID     1.3.6.1.4.1.9590.100.1.2.291121996131431385353...
SeriesInstanceUID    1.3.6.1.4.1.9590.100.1.2.188613955710170417803...
SOPInstanceUID       1.3.6.1.4.1.9590.100.1.2.17333853721263

- Đã lập chỉ mục đủ 10.239 tệp DICOM, không thiếu StudyInstanceUID, SeriesInstanceUID hay SOPInstanceUID theo kết quả đã kiểm tra.

- metadata.csv có 6.775 Series, khớp hoàn toàn với chỉ mục DICOM.

- Cả 3.568 dòng trong bốn CSV đều khớp cặp StudyInstanceUID và SeriesInstanceUID.

- Tên tệp trong CSV là dạng 000000.dcm, còn tên tệp thực tế được tải xuống là UUID. Vì vậy, so sánh tên tệp trực tiếp không phù hợp.

- Chưa xác nhận được liên kết đến đúng ảnh ở cấp từng tệp, đặc biệt là mối quan hệ giữa ảnh toàn cảnh, ảnh crop và ROI mask.

## Kết luận Notebook 02 — Data Quality & Linkage

Đã hoàn thành khảo sát cấu trúc và chất lượng ban đầu của các bảng metadata CBIS-DDSM, kiểm tra khóa định danh, giá trị thiếu, bản ghi trùng lặp và tính nhất quán giữa metadata với chỉ mục DICOM.

Kết quả đối chiếu xác nhận 6.775 `SeriesInstanceUID` trong metadata đều xuất hiện trong chỉ mục DICOM. Tuy nhiên, 10.704 lượt đường dẫn từ các bảng mô tả tổn thương chưa khớp tên tệp DICOM thực tế. Vì vậy, việc ánh xạ từng đường dẫn đến đúng đối tượng ảnh và xác nhận loại ảnh tương ứng chưa được xem là hoàn tất.

Các bảng mô tả tổn thương có trường `pathology`, được dùng làm nguồn nhãn lành tính/ác tính sau khi chuẩn hóa và kiểm tra giá trị thực tế. Các trường đường dẫn ảnh gốc, ảnh crop và ROI mask sẽ được đối chiếu trước khi đưa vào quy trình xây dựng dữ liệu huấn luyện.

**Kết luận:** Hoàn thành kiểm tra chất lượng metadata và tính nhất quán UID ở cấp Series. Việc xác minh ánh xạ ảnh–nhãn ở cấp tổn thương được chuyển sang giai đoạn xây dựng bảng dữ liệu chuẩn; chưa sử dụng các đường dẫn chưa xác minh làm căn cứ huấn luyện.